# Exercise: Feature Engineering Basics

## Do You Want to Build a Snowman?

Prepare this winter data so that a model could predict the height of a snowman. You will use the scikit-learn tools from notebook 01, one at a time.

<img src="assets/olaf.jpeg">

## Learning Objectives

At the end of this exercise, you should be able to:

- Classify each feature as numeric, nominal or binary, and choose a transformation for each type.
- Impute, encode and scale the features of a small dataset with scikit-learn transformers.
- Transform unseen test data with the transformers fitted on the training data.

## Setup

Documentation: [`set_config`](https://scikit-learn.org/stable/modules/generated/sklearn.set_config.html).

In [ ]:
import pandas as pd
from sklearn import set_config
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Transformers return pandas DataFrames instead of NumPy arrays
set_config(transform_output="pandas")

## The Data

Each row is one winter day: the temperature, what you had for lunch and dinner, whether it snowed or rained, and the height of the snowman you built. The target is `height_snowman_cm`.

In [ ]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data)
df_train

The test set below contains four new days. Keep it aside until you have fitted the transformers on the training data: you will then transform it with those fitted transformers.

In [ ]:
# Four new days, kept aside to check the fitted transformers on unseen data
test_data = {
    "temp": [2, None, -4, 6],
    "lunch": ["soup", "burger", "pasta", "sandwich"],
    "dinner": ["pizza", "noodles", None, "fishsticks"],
    "precipitation": ["yes", "no", "yes", "no"],
    "height_snowman_cm": [40, 0, 110, 0],
}

df_test = pd.DataFrame(data=test_data)
df_test

## 1. Separate the Features and the Target

Split `df_train` into the features `X_train` and the target `y_train`.

In [ ]:
target = "height_snowman_cm"

# TODO: X_train holds every column except the target, y_train holds the target
X_train = ...
y_train = ...

## 2. Identify the Variable Types and the Missing Values

Decide for each feature whether it is **numeric**, **nominal** (categories without an order) or **binary** (two categories). The type determines the transformation: numeric features are scaled, and categorical features (nominal and binary) are encoded. Then count the missing values in each column of `X_train`: which columns need imputing?

In [ ]:
# TODO: count the missing values in each column
...

## 3. Impute the Missing Values

Impute `temp`, a numeric variable, with its **median**, and `dinner`, a nominal variable, with its **most frequent** value. Fit each imputer on the training data and transform it in one step with `.fit_transform()`. Select each column with double brackets, for example `X_train[["temp"]]`: scikit-learn transformers expect a two-dimensional input (a DataFrame), not a Series. Documentation: [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html).

In [ ]:
# TODO: impute temp with its median
temp_imputer = SimpleImputer(strategy=...)
temp_imputed_train = ...

# TODO: impute dinner with its most frequent value
dinner_imputer = ...
dinner_imputed_train = ...

## 4. Encode the Categorical Variables

`lunch` and `dinner` are nominal, and `precipitation` is binary. One-hot encode all three, dropping the first category of each: encode `lunch` and `precipitation` together with one encoder, and the imputed `dinner` column with a second encoder. Use `handle_unknown="ignore"`, so that the encoders do not raise an error for categories that appear only in new data, and `sparse_output=False`, so that they return a dense result that can be stored in a DataFrame. Documentation: [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

In [ ]:
# TODO: one-hot encode lunch and precipitation together
categorical_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
lunch_precipitation_encoded_train = ...

# TODO: one-hot encode the imputed dinner column in the same way
dinner_encoder = ...
dinner_encoded_train = ...

## 5. Scale the Numeric Variable

Scale the imputed `temp` column to the range 0 to 1 with [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html).

In [ ]:
# TODO: fit a MinMaxScaler on the imputed temp column and transform it
temp_scaler = ...
temp_scaled_train = ...

## 6. Combine Everything into `X_train_fe`

Place the scaled `temp`, the encoded `lunch` and `precipitation`, and the encoded `dinner` DataFrames side by side with `pd.concat()`. Keep this column order: the test data in the last task is combined in the same order.

In [ ]:
# TODO: concatenate the transformed DataFrames along the columns (axis=1)
X_train_fe = ...
X_train_fe

## 7. Transform the Test Data

Apply the **fitted** imputers, encoders and scaler to the test data, in the same order as for the training data: impute first, then scale or encode. Use only `.transform()`, never `.fit()` or `.fit_transform()`: the test data must not influence what the transformers learned. `X_test_fe` should end up with the same columns as `X_train_fe`.

In [ ]:
X_test = df_test.drop(columns=target)

# TODO: impute, then scale temp with the fitted temp_imputer and temp_scaler
temp_scaled_test = ...

# TODO: impute, then encode dinner; encode lunch and precipitation
dinner_encoded_test = ...
lunch_precipitation_encoded_test = ...

X_test_fe = pd.concat(
    [temp_scaled_test, lunch_precipitation_encoded_test, dinner_encoded_test], axis=1
)
X_test_fe

## Summary

In this exercise you:

- Separated the features from the target.
- Imputed a numeric variable with its median and a nominal variable with its most frequent value.
- One-hot encoded the nominal and binary variables, and scaled the numeric one.
- Transformed unseen test data with the fitted tools only, so the test data had no influence on them.